# JAVDL — Google Colab

Minimal Colab runner for the `javdl` branch. Run the cells from top to bottom after a fresh runtime.

**Required Colab Secrets:** `TELEGRAM_API_ID`, `TELEGRAM_API_HASH`, `TELEGRAM_BOT_TOKEN`.


## 1. Clone the `javdl` branch


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = Path("/content/javdl")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
print(f"JAVDL {BRANCH} ready: {Path.cwd()}")


## 2. Install dependencies


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium


## 3. Load Colab Secrets

Add the three required values in **Colab → Secrets** before running this cell.


In [ ]:
from google.colab import userdata

required = ["TELEGRAM_API_ID", "TELEGRAM_API_HASH", "TELEGRAM_BOT_TOKEN"]
secrets = {}

for name in required:
    try:
        secrets[name] = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read Colab Secret {name!r}: {exc}") from exc

missing = [name for name, value in secrets.items() if not value]
if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

os.environ.update({
    **{name: str(value) for name, value in secrets.items()},
    "TELEGRAM_SESSION": "javdl_colab",
    "TELEGRAM_UPLOAD_SESSION": "javdl_uploads",
    "TELEGRAM_UPLOAD_CLIENTS": "2",
    "TELEGRAM_MAX_CONCURRENT_TRANSMISSIONS": "8",
})

print("Telegram credentials loaded. Upload clients: 2")


## 4. Enable application logs

Keeps extraction, download, thumbnail, FFmpeg and Telegram upload diagnostics visible in Colab.


In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    stream=sys.stdout,
    force=True,
)

for logger_name in ("app", "app.extractor", "app.downloader", "app.uploaders", "app.jobs", "app.bot"):
    logging.getLogger(logger_name).setLevel(logging.INFO)

print("Application logging enabled.")


## 5. Quick import check


In [ ]:
!python -m compileall -q app main.py scripts/colab.py

from app.bot.telegram import create_app

print("JAVDL imports and syntax check passed.")


## 6. Start the Telegram bot

Run this only after the setup cells pass. Colab already has an asyncio event loop, so the bot is started with `await app.start()`.

Send the bot a Javtiful video or channel/studio URL after it starts. Stop the notebook cell with the Colab stop/interrupt control.


In [ ]:
from pyrogram import idle
from app.bot.telegram import create_app

app = create_app()
print("Starting JAVDL Telegram bot…")

await app.start()
print("JAVDL Telegram bot is running.")

try:
    await idle()
finally:
    if app.is_connected:
        await app.stop()
    print("JAVDL Telegram bot stopped.")
